# Recomputing a GMD trajectory with CP2K **pure plane-wave PBE**

This notebook re-computes the **energies and forces** of a GMD *FragmentDuplication*
**OOD test** trajectory at the **PBE** level using a **plane-wave** basis — *not*
Gaussian basis sets — through CP2K's **SIRIUS** backend, driven by the GOAL MD
module (`goal.md.core.calculator_factory`).

**Why SIRIUS / plane waves?** CP2K's default `QUICKSTEP` method is *GPW* (Gaussian
orbitals + a plane-wave density grid) — it still uses a Gaussian basis. To get a
genuinely **basis-set-free, pure plane-wave** DFT (like Quantum ESPRESSO / VASP)
we use CP2K's SIRIUS engine (`METHOD SIRIUS`, `&PW_DFT`) with norm-conserving
**GTH-PBE** pseudopotentials. The GOAL `cp2k` calculator exposes this via
`method="sirius"`.

**No local CP2K needed.** CP2K runs inside the official `cp2k/cp2k` **podman**
container; ASE talks to a containerised `cp2k_shell` through a small wrapper.

**Prerequisites** (one-time):
```bash
podman pull docker.io/cp2k/cp2k:latest   # already done on this machine
```
The wrapper at `$HOME/bin/cp2k_shell` is created/checked in the next cell.

> **Goal of this notebook:** verify the whole pipeline on **one frame**, then
> hand off the full 200-frame trajectory to a background script.

## 1. The podman `cp2k_shell` wrapper

In [ ]:
import os, subprocess, textwrap
from pathlib import Path

WRAPPER = Path.home() / "bin" / "cp2k_shell"

# Self-contained: (re)create the wrapper if it is missing. It makes the
# containerised cp2k_shell behave like a local binary for ASE.
WRAPPER_SRC = r"""#!/usr/bin/env bash
IMAGE="${CP2K_IMAGE:-docker.io/cp2k/cp2k:latest}"
OMP="${CP2K_OMP:-8}"
DATA_DIR="${CP2K_CONTAINER_DATA_DIR:-/opt/cp2k/share/cp2k/data}"
HOSTCWD="$(pwd -P)"
podman run --rm -i \
  --userns=keep-id \
  -v "$HOSTCWD:$HOSTCWD:z" -w "$HOSTCWD" \
  -e OMP_NUM_THREADS="$OMP" \
  -e CP2K_DATA_DIR="$DATA_DIR" \
  "$IMAGE" cp2k_shell "$@" \
  | stdbuf -oL tee -a "${CP2K_SHELL_LOG:-/dev/null}" \
  | grep --line-buffered -E '^[[:space:]]*([*]|CP2K Shell Version:|[-+0-9.])'
"""
if not WRAPPER.exists():
    WRAPPER.parent.mkdir(parents=True, exist_ok=True)
    WRAPPER.write_text(WRAPPER_SRC)
    WRAPPER.chmod(0o755)
    print("created", WRAPPER)
else:
    print("wrapper present:", WRAPPER)

# ASE requires the command string to contain "cp2k_shell"; use the full path.
os.environ["ASE_CP2K_COMMAND"] = str(WRAPPER)
os.environ["CP2K_OMP"] = "16"          # OpenMP threads inside the container
print(WRAPPER.read_text())

### Notes on why the wrapper is not just `podman run … cp2k`

* ASE drives a **persistent** `cp2k_shell` over stdin/stdout pipes, so we need
  `-i` and the command name must contain `cp2k_shell`.
* `-v "$(pwd):$(pwd):z"` bind-mounts the working directory (ASE writes
  `cp2k.inp/out/pos` there). The **`:z`** relabel is required on SELinux-enforcing
  hosts — without it the container gets *Permission denied*.
* The SIRIUS build prints a banner and `[info]/[fft]/…` diagnostics to **stdout**,
  which would corrupt ASE's line protocol; the trailing `grep` **whitelists** only
  genuine protocol lines (`* …`, the version line, and numeric data).

In [ ]:
# Quick handshake: confirm podman + container + wrapper work before any DFT.
out = subprocess.run(
    [str(WRAPPER)], input="VERSION\nEXIT\n",
    capture_output=True, text=True, timeout=180,
).stdout
print(out)
assert "CP2K Shell Version" in out, "cp2k_shell handshake failed"
print("OK — cp2k_shell responds through podman.")

## 2. Load a GMD *FragmentDuplication* OOD trajectory

In [ ]:
import numpy as np
from ase.io import read

# Resolve the project root (folder containing pyproject.toml).
here = Path.cwd()
PROJECT = next((p for p in [here, *here.parents] if (p / "pyproject.toml").exists()), here)

TRAJ = PROJECT / "data/GMD/FragmentDuplication/TestOOD/O4C4H6_PBE.traj"
frames = read(str(TRAJ), index=":")
print(f"{TRAJ.name}: {len(frames)} frames")

FRAME_INDEX = 0
frame = frames[FRAME_INDEX]
ref_e = frame.get_potential_energy()
ref_f = frame.get_forces()
print("formula :", frame.get_chemical_formula(), f"({len(frame)} atoms)")
print("cell (A):", np.round(np.diag(frame.cell), 3), " pbc:", frame.pbc)
print("ref E   :", round(ref_e, 6), "eV   (GMD label: PBE/def2-SVP, all-electron)")
print("ref |F|max:", round(np.abs(ref_f).max(), 4), "eV/A")

## 3. Build the pure plane-wave PBE calculator (via the GOAL MD module)

`method="sirius"` selects CP2K's plane-wave backend. Key knobs:

* `pw_cutoff` / `gk_cutoff` — plane-wave cutoffs in **bohr⁻¹** (`|G|_max`) for the
  density and the wavefunctions. `18 / 9` ≈ **324 / 81 Ry** — well converged for
  H/C/O GTH-PBE. Use `12 / 6` for a fast (~30 s) smoke test.
* `pseudo_potential="GTH-PBE"` — norm-conserving, resolved from `GTH_POTENTIALS`.
* `preset="molecular"` → Γ-point (`kpts=(1,1,1)`), no stress — the molecule sits
  in a ~15 Å box with vacuum, so a single k-point is correct.
* `dispersion="d3bj"` — adds **DFT-D3(BJ)** (2-body, canonical PBE params) as an
  independent additive term to match an ORCA **PBE-D3BJ** reference.

> ⚠️ CP2K's own `&VDW_POTENTIAL` is **silently ignored under `METHOD SIRIUS`**
> (verified: identical energy with/without it). So dispersion is added at the ASE
> level via `simple-dftd3` — physically exact (D3 is geometry-only) and parameter-
> matched to ORCA. Needs `dftd3-python` (`pixi add dftd3-python`).

In [ ]:
from goal.md.core.calculator_factory import CalculatorFactory

# cp2k.inp/out/pos scratch files land in the CWD, which the wrapper bind-mounts.
WORK = PROJECT / "notebooks" / "cp2k_pw_work"
WORK.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)

PW_CUTOFF, GK_CUTOFF = 18.0, 9.0          # production; (12.0, 6.0) = fast smoke

calc = CalculatorFactory.create(
    "cp2k",
    method="sirius",            # pure plane-wave (no Gaussian basis)
    preset="molecular",
    command=os.environ["ASE_CP2K_COMMAND"],
    xc="PBE",                   # -> libxc XC_GGA_X_PBE + XC_GGA_C_PBE
    pseudo_potential="GTH-PBE",
    pw_cutoff=PW_CUTOFF,
    gk_cutoff=GK_CUTOFF,
    smearing_width=0.001,
    dispersion="d3bj",          # DFT-D3(BJ), 2-body, canonical PBE params (= ORCA)
)
print("calculator:", type(calc).__name__, "| method=SIRIUS+D3BJ | pw/gk =", PW_CUTOFF, GK_CUTOFF)

## 4. Verify on ONE frame (the sample run)

In [ ]:
import time
atoms = frame.copy()
atoms.calc = calc
print("running SIRIUS plane-wave PBE single-point ...", flush=True)
t0 = time.time()
e = atoms.get_potential_energy()
f = atoms.get_forces()
dt = time.time() - t0
print(f"done in {dt:.1f}s")

cos = float(np.sum(ref_f * f) / (np.linalg.norm(ref_f) * np.linalg.norm(f)))
print(f"\nrecomputed E : {e:14.6f} eV   (PBE/PW/GTH-PBE, pseudopotential zero)")
print(f"reference  E : {ref_e:14.6f} eV   (PBE/def2-SVP, all-electron zero)")
print(f"|F|max  ref={np.abs(ref_f).max():.4f}  pw={np.abs(f).max():.4f} eV/A")
print(f"force direction cosine(ref, pw) = {cos:.4f}   (1.0 = identical)")

In [ ]:
# The exact CP2K input GOAL generated — note METHOD SIRIUS, the &PW_DFT block,
# and that each &KIND has only a POTENTIAL (no BASIS_SET => no Gaussian basis).
print((WORK / "cp2k.inp").read_text())

### How to read the comparison

* **Absolute energies are *not* comparable.** The reference is *all-electron*
  ORCA PBE-D3BJ; this run is *pseudopotential* plane-wave PBE-D3BJ. The two use
  different energy zeros (the pseudopotential removes core electrons), so they
  differ by thousands of eV by construction. Compare **forces** (a physical
  observable) and **relative** energies. `dump/analyze_pw_vs_ref.py` reports
  the force MAE and the shift-corrected energy MAE for you.
* A force-direction cosine ≈ **0.99** at `pw/gk = 18/9` is the expected level of
  agreement between two different PBE flavours (all-electron Gaussian vs
  pseudopotential plane-wave). At the deliberately-low `12/6` cutoff it drops to
  ~0.91 and force magnitudes are ~2× too large — that is plane-wave
  under-convergence, not a pipeline problem.

> ⚠️ **Energy comparability across frames.** Each GMD frame carries a slightly
> *different* simulation cell. At a fixed plane-wave cutoff the number of plane
> waves depends on the cell, so plane-wave **total energies are only directly
> comparable between frames when the cell is held constant**. On the raw
> per-frame cells the recomputed energy *differences* pick up a spurious
> basis-set-variation term (e.g. a ~6 eV spread where the reference varies < 1 eV).
> **Forces are unaffected.** If you want comparable *relative* energies, pass
> `--fixed-cell` to the batch script (§5): it puts every frame in one common box
> (and is also faster — SIRIUS reuses a single FFT grid).

When you're satisfied the single frame works, close the calculator and launch the
full run below.

In [ ]:
calc.close()
os.chdir(PROJECT)   # back to the project root
print("calculator closed.")

## 5. Recompute the whole trajectory in the background

One frame takes ~1–2.5 min at `pw/gk = 18/9`, so 200 frames is a few hours — run
it detached with `dump/recompute_gmd_cp2k_pw.py`. It streams results to an
extended-XYZ file (keeping *both* the recomputed and the reference E/F),
**resumes** automatically if interrupted, and survives a failed frame.

```bash
cd </path/to/GOAL>          # project root
export ASE_CP2K_COMMAND="$HOME/bin/cp2k_shell" CP2K_OMP=16
mkdir -p logs
nohup pixi run python dump/recompute_gmd_cp2k_pw.py \
    --traj data/GMD/FragmentDuplication/TestOOD/O4C4H6_PBE.traj \
    --pw-cutoff 18 --gk-cutoff 9 \
    > logs/recompute_O4C4H6.log 2>&1 &

tail -f logs/recompute_O4C4H6.log        # watch progress
```

Output: `data/.../TestOOD/O4C4H6_PBE_PWPBE.extxyz` (+ a `.meta.json` with
per-frame timings). Re-running the same command after an interruption continues
from the last written frame. Use `--stop N` to limit frames, `--stride N` to
sub-sample, `--pw-cutoff/--gk-cutoff` to trade accuracy for speed.

**Add `--fixed-cell` if you want comparable relative energies** (see the caveat
in §4): it recomputes every frame in one common box instead of its own, so the
plane-wave basis — and hence the energy zero — is identical across frames:

```bash
    ... --pw-cutoff 18 --gk-cutoff 9 --fixed-cell \
```

## 6. Analyse the recomputed trajectory

In [ ]:
import matplotlib.pyplot as plt

OUT = TRAJ.with_name(f"{TRAJ.stem}_PWPBE.extxyz")
if not OUT.exists():
    print(f"No results yet at {OUT} — launch the background run in section 5 first.")
else:
    rec = read(str(OUT), index=":")
    e_pw  = np.array([a.get_potential_energy() for a in rec])
    e_ref = np.array([a.info["ref_energy"] for a in rec])
    f_pw  = np.concatenate([a.get_forces().ravel() for a in rec])
    f_ref = np.concatenate([a.arrays["ref_forces"].ravel() for a in rec])
    n = len(rec)
    print(f"{n} frames recomputed")

    # Align relative energies (subtract each series' mean) — absolute zeros differ.
    de_pw  = (e_pw  - e_pw.mean())
    de_ref = (e_ref - e_ref.mean())

    fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
    ax[0].plot(de_ref, "o-", label="ref (def2-SVP)", ms=3)
    ax[0].plot(de_pw,  "s-", label="plane-wave", ms=3)
    ax[0].set(xlabel="frame", ylabel="E - <E>  (eV)", title="Relative energy along trajectory")
    ax[0].legend()

    ax[1].scatter(f_ref, f_pw, s=4, alpha=0.4)
    lim = np.abs(np.concatenate([f_ref, f_pw])).max()
    ax[1].plot([-lim, lim], [-lim, lim], "k--", lw=1)
    fmae = np.abs(f_pw - f_ref).mean()
    ax[1].set(xlabel="reference force (eV/A)", ylabel="plane-wave force (eV/A)",
              title=f"Force parity   MAE={fmae:.3f} eV/A")
    fig.tight_layout()
    plt.show()